In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
from itertools import combinations
import json
import os

In [39]:
# ── CHANGE THESE TWO LINES per run ──────────────────────────────
MODEL_FOLDER = "muril-base-cased" # "bert-base-multilingual-cased" or "muril-base-cased" or "xlm-roberta-base"
STATE        = "finetuned" # "pretrained" or "finetuned"
# ────────────────────────────────────────────────────────────────

BASE_PATH  = "/Users/harshaggarwal/Projects_4/hinemo_project/models"
BASE_DIR   = f"{BASE_PATH}/hidden_states_full_means_4k_othwerwise_3k"
MODEL_DIR  = f"{BASE_DIR}/{MODEL_FOLDER}"
OUTPUT_DIR = f"{BASE_PATH}/phase7a_probing_results_4k_lambda/{MODEL_FOLDER}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "joy", "sadness"]

print(f"Model:  {MODEL_FOLDER}")
print(f"State:  {STATE}")

Model:  muril-base-cased
State:  finetuned


In [40]:
hidden = np.load(f"{MODEL_DIR}/hidden_{STATE}_full.npy")
meta   = pd.read_csv(f"{MODEL_DIR}/metadata_full.csv")
meta   = meta.reset_index(drop=True)

lambda_vals = meta["lambda"].values
n_samples, n_layers, hidden_dim = hidden.shape

print(f"Hidden states: {hidden.shape}")
print(f"\nSamples per emotion:")
print(meta["gpt_emotion"].value_counts())

Hidden states: (4000, 12, 768)

Samples per emotion:
gpt_emotion
anger      1000
disgust    1000
joy        1000
sadness    1000
Name: count, dtype: int64


In [41]:
r2_per_emotion  = {}
lsl_per_emotion = {}

for emotion in EMOTIONS:
    mask      = meta["gpt_emotion"] == emotion
    X_emotion = hidden[mask.values]
    y_emotion = lambda_vals[mask.values]

    print(f"\n── {emotion.upper()} ({mask.sum()} samples) ──")

    r2_curve = []
    for layer in range(n_layers):
        X      = X_emotion[:, layer, :]
        probe  = Ridge(alpha=1.0)
        scores = cross_val_score(probe, X, y_emotion, cv=5, scoring="r2")
        r2_curve.append(scores.mean())
        print(f"  Layer {layer+1:2d}: R² = {scores.mean():.4f}")

    lsl_e = int(np.argmax(r2_curve)) + 1
    r2_per_emotion[emotion]  = r2_curve
    lsl_per_emotion[emotion] = lsl_e

    print(f"  LSL({emotion}) = Layer {lsl_e}  |  Peak R² = {max(r2_curve):.4f}")


── ANGER (1000 samples) ──
  Layer  1: R² = 0.0573
  Layer  2: R² = 0.5773
  Layer  3: R² = 0.6120
  Layer  4: R² = 0.5993
  Layer  5: R² = 0.5730
  Layer  6: R² = 0.5423
  Layer  7: R² = 0.2637
  Layer  8: R² = 0.0564
  Layer  9: R² = 0.0653
  Layer 10: R² = 0.0262
  Layer 11: R² = 0.2330
  Layer 12: R² = 0.1606
  LSL(anger) = Layer 3  |  Peak R² = 0.6120

── DISGUST (1000 samples) ──
  Layer  1: R² = 0.0664
  Layer  2: R² = 0.5857
  Layer  3: R² = 0.5969
  Layer  4: R² = 0.5915
  Layer  5: R² = 0.5766
  Layer  6: R² = 0.5636
  Layer  7: R² = 0.2706
  Layer  8: R² = 0.0575
  Layer  9: R² = 0.0794
  Layer 10: R² = 0.0381
  Layer 11: R² = 0.2514
  Layer 12: R² = 0.1511
  LSL(disgust) = Layer 3  |  Peak R² = 0.5969

── JOY (1000 samples) ──
  Layer  1: R² = 0.1020
  Layer  2: R² = 0.6940
  Layer  3: R² = 0.7157
  Layer  4: R² = 0.7178
  Layer  5: R² = 0.7028
  Layer  6: R² = 0.6873
  Layer  7: R² = 0.4064
  Layer  8: R² = 0.0676
  Layer  9: R² = 0.0883
  Layer 10: R² = 0.0610
  Layer 11

In [42]:
print("\n── EMOTION LSL SUMMARY ──")
print(f"{'Emotion':<10} {'LSL':<6} {'Peak R²'}")
print("-" * 30)
for emotion in EMOTIONS:
    print(f"{emotion:<10} {lsl_per_emotion[emotion]:<6} {max(r2_per_emotion[emotion]):.4f}")

pairs          = list(combinations(EMOTIONS, 2))
pairwise_diffs = [abs(lsl_per_emotion[e1] - lsl_per_emotion[e2]) for e1, e2 in pairs]
ELDS           = sum(pairwise_diffs) / len(pairwise_diffs)

print("\n── PAIRWISE LSL DIFFERENCES ──")
for (e1, e2), diff in zip(pairs, pairwise_diffs):
    print(f"  |LSL({e1}) - LSL({e2})| = {diff}")

print(f"\nELDS = {ELDS:.4f}")


── EMOTION LSL SUMMARY ──
Emotion    LSL    Peak R²
------------------------------
anger      3      0.6120
disgust    3      0.5969
joy        4      0.7178
sadness    3      0.7491

── PAIRWISE LSL DIFFERENCES ──
  |LSL(anger) - LSL(disgust)| = 0
  |LSL(anger) - LSL(joy)| = 1
  |LSL(anger) - LSL(sadness)| = 0
  |LSL(disgust) - LSL(joy)| = 1
  |LSL(disgust) - LSL(sadness)| = 0
  |LSL(joy) - LSL(sadness)| = 1

ELDS = 0.5000


In [43]:
results = {
    "model"          : MODEL_FOLDER,
    "state"          : STATE,
    "lsl_per_emotion": lsl_per_emotion,
    "r2_per_emotion" : r2_per_emotion,
    "ELDS"           : ELDS,
    "pairwise_diffs" : {f"{e1}_vs_{e2}": abs(lsl_per_emotion[e1] - lsl_per_emotion[e2]) 
                        for e1, e2 in pairs}
}

save_path = f"{OUTPUT_DIR}/emotion_probing_{STATE}.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {save_path}")

Saved to /Users/harshaggarwal/Projects_4/hinemo_project/models/phase7a_probing_results_4k_lambda/muril-base-cased/emotion_probing_finetuned.json


In [44]:
import json

BASE_PATH = "/Users/harshaggarwal/Projects_4/hinemo_project/models"
models = [
    "bert-base-multilingual-cased",
    "muril-base-cased",
    "xlm-roberta-base"
]
states = ["pretrained", "finetuned"]

print(f"{'Model':<35} {'State':<12} {'ELDS':<8} {'Anger':<6} {'Disgust':<8} {'Joy':<6} {'Sadness'}")
print("-" * 85)

for model in models:
    for state in states:
        path = f"{BASE_PATH}/phase7a_probing_results_4k_lambda/{model}/emotion_probing_{state}.json"
        with open(path) as f:
            r = json.load(f)
        lsl = r["lsl_per_emotion"]
        print(f"{model:<35} {state:<12} {r['ELDS']:<8.4f} {lsl['anger']:<6} {lsl['disgust']:<8} {lsl['joy']:<6} {lsl['sadness']}")

Model                               State        ELDS     Anger  Disgust  Joy    Sadness
-------------------------------------------------------------------------------------
bert-base-multilingual-cased        pretrained   0.6667   4      4        5      5
bert-base-multilingual-cased        finetuned    0.0000   4      4        4      4
muril-base-cased                    pretrained   1.3333   6      4        6      4
muril-base-cased                    finetuned    0.5000   3      3        4      3
xlm-roberta-base                    pretrained   0.0000   12     12       12     12
xlm-roberta-base                    finetuned    0.0000   1      1        1      1


In [45]:
#print all 3k and 4k results together
import json
from itertools import combinations

BASE_PATH = "/Users/harshaggarwal/Projects_4/hinemo_project/models"
models = [
    "bert-base-multilingual-cased",
    "muril-base-cased",
    "xlm-roberta-base"
]
states   = ["pretrained", "finetuned"]
EMOTIONS = ["anger", "disgust", "joy", "sadness"]

def print_global_table(sample_size, folder):
    print(f"\n{'='*70}")
    print(f"GLOBAL λ PROBING — {sample_size}")
    print(f"{'='*70}")
    print(f"{'Model':<35} {'State':<12} {'LSL':<6} {'Peak R²'}")
    print("-" * 65)
    for model in models:
        for state in states:
            path = f"{BASE_PATH}/{folder}/{model}/probing_{state}.json"
            with open(path) as f:
                r = json.load(f)
            print(f"{model:<35} {state:<12} {r['LSL']:<6} {r['peak_r2']:.4f}")

def print_emotion_table(sample_size, folder):
    print(f"\n{'='*90}")
    print(f"EMOTION-CONDITIONED λ PROBING — {sample_size}")
    print(f"{'='*90}")
    print(f"{'Model':<35} {'State':<12} {'ELDS':<8} {'Anger':<6} {'Disgust':<8} {'Joy':<6} {'Sadness'}")
    print("-" * 85)
    for model in models:
        for state in states:
            path = f"{BASE_PATH}/{folder}/{model}/emotion_probing_{state}.json"
            with open(path) as f:
                r = json.load(f)
            lsl = r["lsl_per_emotion"]
            print(f"{model:<35} {state:<12} {r['ELDS']:<8.4f} {lsl['anger']:<6} {lsl['disgust']:<8} {lsl['joy']:<6} {lsl['sadness']}")

# 3k results
print_global_table("3k", "phase7a_probing_results_3k_lambda")
print_emotion_table("3k", "phase7a_probing_results_3k_lambda")

# 4k results
print_global_table("4k", "phase7a_probing_results_4k_lambda")
print_emotion_table("4k", "phase7a_probing_results_4k_lambda")


GLOBAL λ PROBING — 3k
Model                               State        LSL    Peak R²
-----------------------------------------------------------------
bert-base-multilingual-cased        pretrained   5      0.7139
bert-base-multilingual-cased        finetuned    4      0.6970
muril-base-cased                    pretrained   4      0.7615
muril-base-cased                    finetuned    3      0.7120
xlm-roberta-base                    pretrained   12     0.7005
xlm-roberta-base                    finetuned    1      0.6525

EMOTION-CONDITIONED λ PROBING — 3k
Model                               State        ELDS     Anger  Disgust  Joy    Sadness
-------------------------------------------------------------------------------------
bert-base-multilingual-cased        pretrained   0.6667   4      4        5      5
bert-base-multilingual-cased        finetuned    0.0000   4      4        4      4
muril-base-cased                    pretrained   0.0000   4      4        4      4
muril-bas